# SpectraCNN — constellations and per-SNR behaviour

This notebook explores the synthetic I/Q generator and the trained model's noise threshold. It uses `spectracnn.signals` directly and reads the committed evaluation CSV, so it does not require retraining.

Run it from the `notebooks/` directory after `uv pip install -e .`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from spectracnn.signals import MODULATIONS, generate_sample

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
REPORTS = Path("../reports")

## Clean constellations

At 20 dB the four ideal constellations are visible: two BPSK clusters on the I axis, four QPSK points, and the 4×4 / 8×8 grids of 16-QAM and 64-QAM. Pulse shaping and timing offset spread each symbol into a small cloud.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
for ax, modulation in zip(axes.ravel(), MODULATIONS, strict=True):
    frame = generate_sample(modulation, length=1024, snr_db=20.0, rng=rng)
    ax.scatter(frame[0], frame[1], s=4, alpha=0.55, color="#0891b2")
    ax.set_title(f"{modulation} · 20 dB")
    ax.set_xlabel("I")
    ax.set_ylabel("Q")
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
fig.suptitle("Received constellations at high SNR", y=0.98)
fig.tight_layout()
plt.show()

## The noise floor

The same 16-QAM generator at decreasing SNR. Around 0 dB the grid is still distinguishable; below −4 dB the clusters merge and the task becomes close to chance for higher-order QAM.

In [ ]:
snrs = [-6.0, 0.0, 10.0, 20.0]
fig, axes = plt.subplots(1, len(snrs), figsize=(14, 3.8), sharex=True, sharey=True)
for ax, snr in zip(axes, snrs, strict=True):
    frame = generate_sample("16QAM", length=1024, snr_db=snr, rng=rng)
    ax.scatter(frame[0], frame[1], s=4, alpha=0.5, color="#7c3aed")
    ax.set_title(f"{snr:+.0f} dB")
    ax.set_xlabel("I")
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
axes[0].set_ylabel("Q")
fig.suptitle("16-QAM under AWGN", y=1.02)
fig.tight_layout()
plt.show()

## Spectrum and waveform

Root-raised-cosine shaping confines the signal around DC; the carrier offset rotates it and the timing shift delays the waveform. These are the impairments the CNN must learn to ignore.

In [ ]:
frame = generate_sample("QPSK", length=1024, snr_db=10.0, rng=rng)
complex_frame = frame[0] + 1j * frame[1]
spectrum = np.fft.fftshift(np.fft.fft(complex_frame))
frequency = np.fft.fftshift(np.fft.fftfreq(len(complex_frame)))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(frame[0, :200], label="I", linewidth=1.2)
axes[0].plot(frame[1, :200], label="Q", linewidth=1.2)
axes[0].set_title("QPSK waveform (first 200 samples)")
axes[0].set_xlabel("sample")
axes[0].legend()
axes[1].plot(frequency, 20 * np.log10(np.abs(spectrum) + 1e-9), color="#0891b2")
axes[1].set_title("Normalized spectrum")
axes[1].set_xlabel("normalized frequency")
axes[1].set_ylabel("dB")
fig.tight_layout()
plt.show()

## Accuracy vs SNR

The committed `reports/per_snr.csv` is produced by `spectracnn-train` on fresh frames at each SNR. The 25% dashed line is the random baseline for four balanced classes.

In [ ]:
path = REPORTS / "per_snr.csv"
if path.exists():
    curve = pd.read_csv(path)
    display(curve)
    ax = curve.plot(x="snr_db", y=["accuracy", "macro_f1"], marker="o", figsize=(8, 4.5))
    ax.axhline(0.25, color="#94a3b8", linestyle="--", linewidth=1, label="random")
    ax.set_xlabel("SNR (dB)")
    ax.set_ylabel("score")
    ax.set_ylim(0, 1.05)
    ax.set_title("SpectraCNN performance vs SNR")
    ax.legend()
    plt.show()
else:
    print("reports/per_snr.csv not found; run `uv run spectracnn-train` first.")

## Takeaway

The generator makes the learning problem explicit: constellation geometry, pulse shaping and impairments are all controlled. The per-SNR curve is the honest summary — aggregate accuracy over a mixed-SNR test set hides the low-SNR collapse that any real receiver must handle.